In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cvxpy as cp
from dotenv import load_dotenv
from tiingo import TiingoClient
from fredapi import Fred
from tabulate import tabulate

load_dotenv()

# --- Universe --- 
BENCHMARKS = ['SPY', 'RSP']  #   Cap vs equal weight S&p ETFs
ASSETS  = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']  # sector ETFs
FACTORS = ['VTI', 'VV', 'VBR', 'VBK', 'VTV', 'VUG']  # 3factor: VTI (total market), VV (volatility), VBR (value), VBK (growth), VTV (value), VUG (growth)
TICKERS=  BENCHMARKS + ASSETS + FACTORS

FREQ = '1hour'

# --- Date windows ---
FETCH_START = '2020-01-01'   # PLTR IPO'd 2020-09-30; gives a clean Nov-2020 start
FETCH_END   = '2026-05-30'

    # Witholding the training and testing windows for now
#TRAIN_START = '2020-11-01'
#TRAIN_END   = '2025-04-30'
#HOLD_START  = '2025-05-01'
#HOLD_END    = '2026-04-30'

# --- Local paths ---
DATA_DIR        = 'data'
RAW_CACHE       = f'{DATA_DIR}/intraday_1hr_raw.parquet'
CLEANED_CACHE = f'{DATA_DIR}/intraday_1hr_cleaned.parquet'

os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
# Ticker Descriptions from Metadata Enpoint

metadata = {}

if os.path.exists(f'{DATA_DIR}/ticker_metadata.parquet'):
    metadata_df = pd.read_parquet(f'{DATA_DIR}/ticker_metadata.parquet') 
    print("Loaded metadata from cache.")
    print(metadata_df)
else:
    for ticker in TICKERS:
        client = TiingoClient({'api_key': os.getenv('TIINGO_API_KEY')})
        metadata[ticker] = client.get_ticker_metadata(ticker)['name']
        metadata_df = pd.DataFrame.from_dict(metadata, orient='index', columns=['Description'])
        metadata_df.to_parquet(f'{DATA_DIR}/ticker_metadata.parquet')
        print("Fetched metadata from Tiingo API and saved to cache.")
        print(metadata_df)



Loaded metadata from cache.
                                            Description
SPY                              SPDR S&P 500 ETF Trust
RSP                   INVESCO S&P 500 EQUAL WEIGHT ETF 
XLK   STATE STREET(R) TECHNOLOGY SELECT SECTOR SPDR(...
XLF   STATE STREET(R) FINANCIAL SELECT SECTOR SPDR(R...
XLE   STATE STREET(R) ENERGY SELECT SECTOR SPDR(R) ETF 
XLV   STATE STREET(R) HEALTH CARE SELECT SECTOR SPDR...
XLI   STATE STREET(R) INDUSTRIAL SELECT SECTOR SPDR(...
XLY   STATE STREET(R) CONSUMER DISCRETIONARY SELECT ...
XLP   STATE STREET(R) CONSUMER STAPLES SELECT SECTOR...
XLU   STATE STREET(R) UTILITIES SELECT SECTOR SPDR(R...
XLB   STATE STREET(R) MATERIALS SELECT SECTOR SPDR(R...
XLRE  STATE STREET(R) REAL ESTATE SELECT SECTOR SPDR...
XLC   STATE STREET(R) COMMUNICATION SERVICES SELECT ...
VTI   VANGUARD TOTAL STOCK MARKET INDEX FUND ETF SHARES
VV             VANGUARD LARGE-CAP INDEX FUND ETF SHARES
VBR      VANGUARD SMALL-CAP VALUE INDEX FUND ETF SHARES
VBK     VANGUARD SMA

In [ ]:
# === Reading data from Tiingo / Cache ===

if os.path.exists(RAW_CACHE):
    print(f"Loading raw cache: {RAW_CACHE}")
    raw = pd.read_parquet(RAW_CACHE)
else:
    print("Fetching data from Tiingo...")
    client = TiingoClient({'api_key': os.getenv('TIINGO_API_KEY')})

    dataframes = []

    for ticker in TICKERS:
        print(f"Fetching {ticker}...")

        df = client.get_dataframe(
            ticker, 
            startDate=FETCH_START, 
            endDate=FETCH_END, 
            frequency=FREQ,
            columns='open,high,low,close,volume')
            #afterHours=True, --- Not available on the standard endpoint; requires requests
            #forceFill=False)
        
        df['ticker'] = ticker
        df = df.reset_index(names='datetime')

        dataframes.append(df)

    raw = pd.concat(dataframes, ignore_index=True)
    raw.to_parquet(RAW_CACHE, index=False) 

raw['datetime'] = raw['datetime'].dt.tz_convert('America/New_York').dt.tz_localize(None)

Fetching data from Tiingo...
Fetching SPY...
Fetching RSP...
Fetching XLK...
Fetching XLF...
Fetching XLE...
Fetching XLV...
Fetching XLI...
Fetching XLY...
Fetching XLP...
Fetching XLU...
Fetching XLB...
Fetching XLRE...
Fetching XLC...
Fetching VTI...
Fetching VV...
Fetching VBR...
Fetching VBK...
Fetching VTV...
Fetching VUG...


In [14]:
# === Initial Intraday Summary Stats  for Data Validation ===

print(f"Data fetched from {FETCH_START} to {FETCH_END}")
print(raw['datetime'].min(), raw['datetime'].max())
print(f"======================================")
print(f"Number of assets: {len(raw['ticker'].unique())}")
print(f"======================================")
print(raw.head())
print(f"======================================")
print(raw.info())
print(f"======================================")
raw.describe()

Data fetched from 2020-01-01 to 2026-05-30
2020-01-09 12:00:00 2026-05-29 15:00:00
Number of assets: 19
             datetime     open     high      low    close    volume ticker
0 2020-01-09 12:00:00  326.515  326.615  325.515  326.050  118642.0    SPY
1 2020-01-09 13:00:00  326.045  326.400  325.730  326.200  109561.0    SPY
2 2020-01-09 14:00:00  326.180  326.750  326.160  326.485   74536.0    SPY
3 2020-01-09 15:00:00  326.480  326.680  326.105  326.600  228786.0    SPY
4 2020-01-10 10:00:00  327.155  327.465  326.900  327.195  143860.0    SPY
<class 'pandas.DataFrame'>
RangeIndex: 190000 entries, 0 to 189999
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   datetime  190000 non-null  datetime64[us]
 1   open      190000 non-null  float64       
 2   high      190000 non-null  float64       
 3   low       190000 non-null  float64       
 4   close     190000 non-null  float64       
 5   volume    190

,datetime,open,high,low,close,volume
count,190000,190000.000000,190000.000000,190000.000000,190000.000000,1.900000e+05
mean,2023-03-21 08:28:58.079999,156.870224,157.193386,156.533641,156.872440,3.736372e+04
min,2020-01-09 12:00:00,17.635000,17.930000,17.495000,17.620000,0.000000e+00
25%,2021-08-15 17:15:00,75.955000,76.150000,75.750000,75.950000,1.053000e+03
50%,2023-03-21 13:30:00,135.570000,135.865000,135.310000,135.580000,9.720000e+03
75%,2024-10-24 11:15:00,201.666250,202.075000,201.312500,201.630000,3.652625e+04
max,2026-05-29 15:00:00,757.800000,758.075000,756.335000,756.810000,6.040482e+06
std,NaN,113.356738,113.551342,113.148553,113.358931,8.160742e+04


In [11]:
# === Wide Pivoting Intraday ===

close_wide = raw.pivot(index='datetime', columns='ticker', values='close')
returns_wide = np.log(close_wide).diff() #<--- Property of log returns: log(a/b) = log(a) - log(b)
simple_returns_wide = close_wide.pct_change() #<--- Simple returns: (a-b)/b 

In [15]:
# --- Summary Stats for Prices and Returns ---

# Returns
extra_stats = pd.DataFrame({
    'skew': returns_wide.skew(),
    'kurtosis': returns_wide.kurtosis()
})

full_stats = pd.concat([returns_wide.describe().T, extra_stats], axis =1)    

print("Returns Summary Statistics:\n")
print(tabulate(full_stats, headers='keys', tablefmt='pipe', floatfmt='.4f'))
print("\n\n")

# Prices

extra_stats = pd.DataFrame({
    'skew': close_wide.skew(),
    'kurtosis': close_wide.kurtosis()
})

full_stats = pd.concat([close_wide.describe().T, extra_stats], axis =1)    

print("Prices Summary Statistics:\n")
print(tabulate(full_stats, headers='keys', tablefmt='pipe', floatfmt='.4f'))

Returns Summary Statistics:

| ticker   |     count |    mean |    std |     min |     25% |    50% |    75% |    max |     skew |   kurtosis |
|:---------|----------:|--------:|-------:|--------:|--------:|-------:|-------:|-------:|---------:|-----------:|
| RSP      | 9999.0000 |  0.0001 | 0.0051 | -0.0745 | -0.0014 | 0.0000 | 0.0016 | 0.0729 |  -0.3656 |    35.7060 |
| SPY      | 9999.0000 |  0.0001 | 0.0049 | -0.0773 | -0.0013 | 0.0001 | 0.0016 | 0.0650 |  -0.6303 |    30.7450 |
| VBK      | 9999.0000 |  0.0001 | 0.0065 | -0.1052 | -0.0018 | 0.0000 | 0.0021 | 0.0835 |  -0.6607 |    23.0490 |
| VBR      | 9999.0000 |  0.0001 | 0.0063 | -0.0936 | -0.0017 | 0.0000 | 0.0019 | 0.0770 |  -0.7568 |    31.6823 |
| VTI      | 9999.0000 |  0.0001 | 0.0050 | -0.0818 | -0.0014 | 0.0001 | 0.0016 | 0.0682 |  -0.7287 |    31.7592 |
| VTV      | 9999.0000 |  0.0001 | 0.0046 | -0.0730 | -0.0013 | 0.0000 | 0.0014 | 0.0724 |  -0.7987 |    38.6536 |
| VUG      | 9999.0000 | -0.0001 | 0.0189 | -1.7927

In [16]:
# === Grabbing EOD data for 2 reasons: 
    # 1) adjusting for splits by backing out the split factor, which IEX doesn't do on the intraday endpoint, and 
    # 2) to further confirm the FRENCH proxy validity by comparing the intraday values to the EOD values

eod_cache = f'{DATA_DIR}/eod_cache.parquet'
if os.path.exists(eod_cache):
    print(f"Loading EOD cache: {eod_cache}")
    eod = pd.read_parquet(eod_cache)
else:
    print("Fetching EOD data from Tiingo...")
    eod_dataframes = []

    for ticker in TICKERS:
        print(f"Fetching EOD {ticker}...")
        client = TiingoClient({'api_key': os.getenv('TIINGO_API_KEY')})
        df = client.get_dataframe(
            ticker, 
            startDate=FETCH_START, 
            endDate=FETCH_END, 
            frequency='daily',
            columns='close,adjClose,volume')
        
        df['ticker'] = ticker
        df = df.reset_index(names='datetime')

        eod_dataframes.append(df)

    eod = pd.concat(eod_dataframes, ignore_index=True)
    eod.to_parquet(eod_cache, index=False)

eod.describe()

Loading EOD cache: data/eod_cache.parquet


,close,adjClose,volume
count,30590.000000,30590.000000,3.059000e+04
mean,156.644927,123.952333,1.334053e+07
std,113.201715,109.238829,2.268006e+07
min,17.660000,9.169744,6.888600e+04
25%,75.910000,43.619887,2.205373e+06
50%,135.360000,90.688839,5.876264e+06
75%,201.355000,161.102355,1.267310e+07
max,756.480000,754.556559,3.896121e+08


In [ ]:
# === Divisor Constrcution for Adjusting Intraday Data

# EOD Pivoting and Cleaning: localized to none because EOD uses EST while intraday uses UTC

eod['date'] = (pd.to_datetime(eod['datetime'], utc=True)
                 .dt.tz_localize(None)
                 .dt.normalize())

eod_close = eod.pivot(index='date', columns='ticker', values='close')
eod_adj   = eod.pivot(index='date', columns='ticker', values='adjClose')


divisor = eod_close / eod_adj

# Known  Split Date Validation
print(f"======================================")
print(f"2021-04-21: PLTR 1:4 split, divisor ~0.25")
print(f"\n")
print(divisor['VUG'].loc['2026-04-16':'2026-04-23'].round(4))   # ~6.0 → 1.0 at Apr 21
print(f"======================================")
print(f"2025-12-05: XLK 2:1 split, divisor ~2.0")
print(f"\n")
print(divisor['XLK'].loc['2025-12-02':'2025-12-09'].round(4))   # ~2.0 → ~1.0 at Dec 5
print(f"======================================")
print(divisor.iloc[-1].round(6))                                # last row: exactly 1.0, all 19
print(f"======================================")
print((divisor < 1 - 1e-9).any().any())                         # False for this universe

In [ ]:
# === Plotting Prices and Returns by Asset ===

import numpy as np
import matplotlib.colors as mcolors

color_index = range(len(returns_wide.index))

fig, axes = plt.subplots(len(ASSETS), 3, figsize=(30, 6 * len(ASSETS)))
for i, asset in enumerate(ASSETS):

                        
    x_color = returns_wide.index
    y_color = returns_wide[asset]
    color_array = np.arange(len(y_color))

    is_pos = y_color > 0
    is_neg = y_color < 0 
    is_zero = y_color == 0 


    axes[i, 0].plot(close_wide.index, close_wide[asset])
    axes[i, 0].set_title(f'{asset} Price')
    axes[i, 0].xaxis.set_major_locator(mticker.MaxNLocator(5))
    axes[i, 0].set_facecolor('gray')
    axes[i, 0].grid(True, linestyle='--', alpha=0.5)
    
    axes[i, 1].scatter(
                       x_color[is_pos],
                       y_color[is_pos],
                       label='Log Returns',
                       s=1,
                       c=color_array[is_pos],
                       edgecolor='black',
                       linewidths=0.1,
                       cmap='Blues')
    axes[i, 1].scatter(
                       x_color[is_neg],
                       y_color[is_neg],
                       label='Log Returns',
                       s=1,
                       c=color_array[is_neg],
                       edgecolor='black',
                       linewidths=0.1,
                       cmap='Reds')
    axes[i, 1].scatter(
                       x_color[is_zero],
                       y_color[is_zero],
                       label='Log Returns',
                       s=1,
                       edgecolor='black',
                       linewidths=0.1,
                       color='#f0f0f0')
    axes[i, 1].set_title(f'{asset} Log Returns')
    axes[i, 1].xaxis.set_major_locator(mticker.MaxNLocator(5))
    axes[i, 1].set_facecolor('gray')
    axes[i, 1].grid(True, linestyle='--', alpha=0.5)
    axes[i, 1].margins(x=0.01, y=0.01)
    
    axes[i, 2].scatter(
                       x_color[is_pos],
                       y_color[is_pos],
                       label='Log Returns',
                       s=2.5,
                       c=color_array[is_pos],
                       edgecolor='black',
                       linewidths=0.1,
                       cmap='Blues')
    axes[i, 2].scatter(
                       x_color[is_neg],
                       y_color[is_neg],
                       label='Log Returns',
                       s=2.5,
                       c=color_array[is_neg],
                       edgecolor='black',
                       linewidths=0.1,
                       cmap='Reds')
    axes[i, 2].scatter(
                       x_color[is_zero],
                       y_color[is_zero],
                       label='Log Returns',
                       s=2.5,
                       edgecolor='black',
                       linewidths=0.1,
                       color='#f0f0f0')
    axes[i, 2].set_title(f'{asset} Log Returns' + '(Symmetric Log Scale)')
    axes[i, 2].xaxis.set_major_locator(mticker.MaxNLocator(5))
    axes[i, 2].set_facecolor('gray')
    axes[i, 2].grid(True, linestyle='--', alpha=0.5)
    axes[i, 2].margins(x=0.01, y=0.01)
    axes[i, 2].set_yscale('symlog', linthresh=1e-4)  # Set y-axis to symmetric log scale with a threshold of 0.0001


In [ ]:
# === Histograms of Returns ===
import numpy as np
from scipy import stats


fig, axes = plt.subplots(len(ASSETS), 3, figsize=(30, 6 * len(ASSETS)))
for i, asset in enumerate(ASSETS):

    mu = returns_wide[asset].mean()
    sigma = returns_wide[asset].std()
    x_norm = np.linspace(returns_wide[asset].min(), returns_wide[asset].max(), len(returns_wide[asset]))
    y_norm = stats.norm.pdf(x_norm, mu, sigma)
    #y_norm = np.clip(y_norm, a_min=1e-10, a_max=None) # Clipping to prevent log(0) issues in the histogram with log scale
    ln_y_norm = np.log(y_norm)


    axes[i,0].hist(returns_wide[asset].dropna(), 
                 bins=100, 
                 density=True, 
                 alpha=0.7, 
                 #log=True,
                 edgecolor='black',
                 color='red')
    axes[i,0].plot(x_norm, y_norm, 'b--', linewidth=2, label='Normal Distribution Fit')
    axes[i,0].legend()
    axes[i,0].set_title(f'{asset} Log Returns Distribution')
    axes[i,0].xaxis.set_major_locator(mticker.MaxNLocator(10))

    axes[i,1].hist(returns_wide[asset].dropna(), 
                 bins=100, 
                 density=True, 
                 alpha=0.7, 
                 log=True,
                 edgecolor='black', 
                 color='red')
    #axes[i,1].autoscale(enable=False, axis='y')
    axes[i,1].plot(x_norm, y_norm, 'b--', linewidth=2, label='Normal Distribution Fit')
    axes[i,1].legend()
    axes[i,1].set_title(f'{asset} Log Returns Distribution' + '(Log Scale)')
    axes[i,1].xaxis.set_major_locator(mticker.MaxNLocator(10))
    axes[i,1].set_ylim(bottom=1e-10, top=10e3)  
    axes[i,1].margins(x=0, y=0)


    stats.probplot(returns_wide[asset], dist='norm', plot=axes[i,2])
    axes[i,2].set_title(f'{asset} Return Quantile Plot vs Normal')

# Other Possible Summary Stats

## 1. **Returns vs. Prices** - does the variance and magnitude of returns vary with price?

## 2. **Return $\sigma$ vs. Magnitude** - does the variance of returns increase as return magnitude increases?

## 3. **Return $\sigma$ vs. Time** - does the variance change dependent on time? Correlation may also be informative

In [ ]:
# === Subsetting dataframes for each component
returns_wide = returns_wide.dropna()  # Drop rows with NaN values to ensure clean data for analysis
sector_returns  = returns_wide[ASSETS]
factor_returns = returns_wide[FACTORS] 
benchmark_returns = returns_wide[BENCHMARKS]


#  --- Constructing factors
#  Following French 2x3 / 2x2 sorting methodology for SMB and HML

factor_returns['MKT'] = factor_returns['VTI']
factor_returns['SMB'] = 0.5*(factor_returns['VBR'] + factor_returns['VBK']) - 0.5*(factor_returns['VTV'] + factor_returns['VUG'])
factor_returns['HML'] = 0.5*(factor_returns['VBR'] + factor_returns['VTV']) - 0.5*(factor_returns['VBK'] + factor_returns['VUG'])
factor_returns['VOL'] = factor_returns['VV']  # Volatility factor

factor_returns = factor_returns[['MKT', 'SMB', 'HML', 'VOL']]

factor_returns.describe()


In [ ]:
french  = pd.read_csv('/mnt/hard_storage/GithubRepos/project-dev/ssm-finance/data/F-F_Research_Data_Factors_daily.csv', skiprows=3)

french['date'] = pd.to_datetime(french['Unnamed: 0'], format='%Y%m%d', errors='coerce')
french = french.drop(columns=['Unnamed: 0'])
french['date'] = french['date'].dt.tz_localize('America/New_York')
french['date'] = french['date'].dt.tz_localize(None)
french = french.set_index('date')
french  = french / 100  # Convert percentages to decimal form


french  =  french[french.index >= returns_wide.index.min()]  # Filter to match the returns_wide index range
french.describe()

In [ ]:
#  --- Correlationn Between French and Constructed French factors
# Resamples hourly data to daily and aligns

daily_factor_returns = factor_returns.resample('D').sum()
daily_factor_returns.index = daily_factor_returns.index.normalize() 
daily_factor_returns = daily_factor_returns.dropna()


# Validate 
print(f"Intersection - Should be non empty: {daily_factor_returns.index.intersection(french.index)}")

In [ ]:
french['MKT'] = french['Mkt-RF'] + french['RF']  # Reconstruct MKT from Mkt-RF and RF

print(f"MKT vs MKT: {daily_factor_returns['MKT'].corr(french['MKT']):.4f}")
print(f"MKT vs Mkt-RF: {daily_factor_returns['MKT'].corr(french['Mkt-RF']):.4f}")
print(f"SMB vs SMB:    {daily_factor_returns['SMB'].corr(french['SMB']):.4f}")
print(f"HML vs HML:    {daily_factor_returns['HML'].corr(french['HML']):.4f}")

# Reviewing French Factors

```
SMB    = 1/3(SV + SN + SG) − 1/3(BV + BN + BG)
HML    = 1/2(SV + BV) − 1/2(SG + BG)
Mkt−RF = VW return of the entire CRSP universe − 1-month T-bill
```

*Per Claude:*

At the end of June each year t, every eligible stock is independently sorted two ways:

- Size = market cap at end of June t. The breakpoint is the NYSE median — computed from NYSE stocks only, then applied to the whole universe. Below → Small, above → Big.
- Value = BE/ME, where BE is book equity for the fiscal year ending in calendar year t−1 and ME is market cap at December t−1. Breakpoints are the NYSE 30th and 70th percentiles, giving Growth / Neutral / Value. Negative-BE firms are excluded.

The intersection gives six value-weighted portfolios (SG, SN, SV, BG, BN, BV), held from July t through June t+1, reconstituted only once a year. The daily factors are just the daily value-weighted returns of those same annually-formed portfolios


**SMB** - *Small - Big*  - This is the difference between the returns of smallest and biggest stocks measured by market cap around the NYSE median
**HML** - *High - Low* - Returns of High Book / Market stocks minus low book/market stocks
   - In $\frac{BE}{ME}$ as $BE \uparrow$, the stock becomes higher and as $ME \downarrow$ the stock becomes higher
   - As book equity increases, the firm has more capital is theoretically less *'growth'* 
   - As market equity increases relative to book equity, the firm is more speculative and less fundamental

In [ ]:
# 3 Factor CAPM Regression Model
# SPY 

X  = sm.add_constant(pd.concat([benchmark_returns, factor_returns], axis=1))
model =  sm.OLS(benchmark_returns['SPY'], X[['const', 'MKT', 'SMB', 'HML']]).fit(cov_type='HC3')
print(model.summary())

# RSP - Equal Weight S&P 500 ETF
X  = sm.add_constant(pd.concat([benchmark_returns, factor_returns], axis=1))
model_rsp =  sm.OLS(benchmark_returns['RSP'], X[['const', 'MKT', 'SMB', 'HML']]).fit(cov_type='HC3')
print(model_rsp.summary())

In [ ]:
# --- Asset 3 Factor CAPM Regressions

for asset in ASSETS:
    X  = sm.add_constant(pd.concat([sector_returns[asset], factor_returns], axis=1))
    model_asset =  sm.OLS(sector_returns[asset], X[['const', 'MKT', 'SMB', 'HML']]).fit(cov_type='HC3')
    print(f"3 Factor CAPM Regression for {asset}:")
    print(model_asset.summary())
    print("\n\n")

# Estimation Plan (from here)


### The above compiles a reasonable, non-exhaustive set of summary statistics and benchmark information for the S&P sector ETF asset universe using ~6 years of hourly returns

- The IEX window drops all overnight observations and standardizes trading hours from 10:00am - 3:00 pm
- Factors constructed using French's methodology, correlation between daily and hourly constructions above
    - Repair / validate pipeline
    - Grab daily values
    
- Reconsider return measurement: $Close_t - Close_{t-1}$ vs $Open_t - Close_{t}$ with $Close_t - Open_{t+1}$
    - The latter gives something that feels more informative about hourly movement, theoretically the second component will be very small on the hour (tick level) except overnight

    - **Something doesn't feel right about the return construction currently as specified (Close-Close)**

---

### Modeling

- Arima / GARCH - Time-series Type estimates

- NNs 
    - Naive type with time encoded as a feature, no TS-CV
    - TS inclusive (attention / TS aware NNs)

- I feel like I'm missing something important for all of this exercise